# [LAB-04] 03. 분석용 데이터 전처리


## EDA의 결론



| 채택여부 | 변수 | 검정방법 | 유의확률 | 방향 | 공선성 신호 | 근거 |
|---|---|---|---|---|---|---|
| ✅ 채택 | Title | 교차분석 | < 0.001 | Mrs > Miss > Master > Officer > Mr | — | V = 0.697 (Strong) |
| ✅ 채택 | Pclass | 교차분석 | < 0.001 | 1등급 > 2등급 > 3등급 | — | V = 0.262 (Weak) |
| ✅ 채택 | HasCabin | 교차분석 | < 0.001 | 있음 > 없음 | 있음의 86.7%가 1등석 | V = 0.254 (Weak) |
| ✅ 채택 | Embarked | 교차분석 | < 0.001 | C > Q > S | 항구별 Pclass 구성 차이 | V = 0.137 (Weak) |
| ✅ 채택 | Fare | Mann-Whitney U | < 0.001 | 생존 > 사망 | X (VIF 1.12 · FamilySize와 ρ 0.52) | 평균 2배 · 중앙값 2.5배 차이 |
| ✅ 채택 | FamilySize | Mann-Whitney U | < 0.001 | 생존 > 사망 (역U자) | X (VIF 1.13) | 순위 분포 차이 유의 |
| 🟡 보류 | Age | Mann-Whitney U | 0.116 | — | X (VIF 1.11) | 단독 차이는 없으나 연령효과 가능성 |



- 연속형 3종 사이의 공선성은 없다. 상관계수(Spearman)는 최대 약 0.524(Fare–FamilySize)이며, VIF는 세 변수 모두 약 1.1 수준이다.
- 명목형끼리의 겹침은 교차표로만 확인했으므로 정량 판단이 아니다. HasCabin·Embarked가 Pclass와 얼마나 겹치는지는 모형에 함께 넣었을 때 계수와 표준오차의 변화로 판단한다.
- 이 데이터셋은 품질점검 단계에서 이미 정리된 버전이다. PassengerId, Ticket, Name, Cabin, SibSp, Parch, Sex는 제외되어 있다.


## #01. 준비작업

### 1. 라이브러리 참조


In [ ]:
from hossam import load_data
from helpers import *
from IPython.display import display, Markdown


### 2. 데이터 불러오기


In [ ]:
origin = load_data("titanic_qtcheck")
num_desc = load_data("titanic_numerical_summary")
cat_desc = load_data("titanic_categorical_summary")


### 3. 타입변환 (범주형 확정)


In [ ]:
df = my_qtcheck.set_type(origin, as_category=cat_desc.columns)
df.info()


### 4. 컬럼 의미를 정리한 딕셔너리


In [ ]:
column_means = {
    "Survived":   "생존 여부 (0=사망, 1=생존)",
    "Pclass":     "객실 등급 (1=1등급, 2=2등급, 3=3등급)",
    "Age":        "나이",
    "Fare":       "요금",
    "Embarked":   "탑승 항구 (C=Cherbourg, Q=Queenstown, S=Southampton)",
    "HasCabin":   "선실 존재 여부 (True=선실 있음, False=선실 없음)",
    "FamilySize": "동승 가족 수 (SibSp + Parch + 본인)",
    "Title":      "이름에서 추출한 호칭 (Mr/Mrs/Miss/Master/Officer)",
}

column_means


### 5. 변수 유형 분류


In [ ]:
target = "Survived"
target_is_continuous = False

exclude_cols = []

nominal_cols = my_qtcheck.get_categorical_column_names(df)
continuous_cols = my_qtcheck.get_number_column_names(df)

if target_is_continuous:
    continuous_cols.remove(target)
else:
    nominal_cols.remove(target)

continuous_cols = [c for c in continuous_cols if c not in exclude_cols]
nominal_cols = [c for c in nominal_cols if c not in exclude_cols]

print("종속변수 :", target)
print("종속변수 유형:", "연속형" if target_is_continuous else "범주형")
print("범주형   :", nominal_cols)
print("연속형   :", continuous_cols)
print("제외     :", exclude_cols)


## #02. 로그 변환 및 데이터 라벨링

### 1. 변환 대상 확인


기술통계량 표에서 로그 변환 필요성 여부만 확인한다.


In [ ]:
num_desc["log_need"].T


### 2. 로그 변환 대상 추출


In [ ]:
# log_need 판정에 따라 변환 대상을 분류한다.
log_cols = num_desc[num_desc["log_need"] == "log"].index.tolist()
log1p_cols = num_desc[num_desc["log_need"] == "log1p"].index.tolist()
reflect_cols = num_desc[num_desc["log_need"] == "reverse_log1p"].index.tolist()

print("log(x)      :", log_cols)
print("log(1+x)    :", log1p_cols)
print("반사 후 log :", reflect_cols)


### 3. 로그변환 수행


In [ ]:
df_log = my_prep.log_transform(
    origin,
    log_columns=log_cols,
    log1p_columns=log1p_cols,
    reflect_columns=reflect_cols
)


### 4. 범주형 라벨링


In [ ]:
df_label = my_prep.labeling(df_log, columns=nominal_cols)
df_label[nominal_cols].head()


### 5. 최종 데이터셋 저장


In [ ]:
df_label.to_excel("titanic_features.xlsx", index=False)

print("저장 완료 :", df_label.shape)
df_label.head()
